# Phase 1 — Centralized TB Classifier (Colab)

Trains a DenseNet121 TB classifier on **Shenzhen + Montgomery** and locks in the metric harness.

**Before running:** set Runtime → Change runtime type → **GPU**.

The data is downloaded **directly inside Colab** straight into your Google Drive (one-time).
No need to download locally or upload from your machine.

## 1. Mount Drive + clone the repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone your repo (replace with your remote if private — use a token or GitHub auth).
!git clone https://github.com/karansharma1732/project-dheeru-ki-phd.git
%cd project-dheeru-ki-phd/code

## 2. Download datasets into Drive (one-time)

Downloads Montgomery (588 MB) + Shenzhen (3.6 GB) from the NLM mirror straight into
`MyDrive/tb-fl/data/raw/`. Colab's fast connection makes this quick. The cell **skips**
any dataset already present, so re-running in a later session is instant.

In [ ]:
import os, zipfile, urllib.request

RAW = '/content/drive/MyDrive/tb-fl/data/raw'
os.makedirs(RAW, exist_ok=True)

DATASETS = {
    'montgomery': 'https://openi.nlm.nih.gov/imgs/collections/NLM-MontgomeryCXRSet.zip',
    'shenzhen':   'https://openi.nlm.nih.gov/imgs/collections/ChinaSet_AllFiles.zip',
}

def has_data(folder):
    # Considered present if a ClinicalReadings dir with .txt files exists somewhere inside.
    for root, _, files in os.walk(folder):
        if root.endswith('ClinicalReadings') and any(f.endswith('.txt') for f in files):
            return True
    return False

for name, url in DATASETS.items():
    dest = os.path.join(RAW, name)
    if os.path.isdir(dest) and has_data(dest):
        print(f'[skip] {name}: already present at {dest}')
        continue
    os.makedirs(dest, exist_ok=True)
    zip_path = f'/content/{name}.zip'
    print(f'[get ] {name}: downloading ...')
    urllib.request.urlretrieve(url, zip_path)
    print(f'[ok  ] {name}: {os.path.getsize(zip_path)/1e6:.0f} MB — extracting into Drive ...')
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(dest)
    os.remove(zip_path)
    print(f'[done] {name}: extracted to {dest}')

print('\nContents of', RAW)
!ls -R "{RAW}" | head -40

## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 4. Check config paths
Confirm the dataset paths in `config.yaml` match the Drive folders above. Adjust epochs /
batch size for the Colab GPU if needed.

In [ ]:
import yaml
cfg = yaml.safe_load(open('config.yaml'))
cfg['paths']

## 5. Parse demographics → demographics.csv

In [ ]:
!python parse_demographics.py --config config.yaml

In [ ]:
# Quick look at class + subgroup balance.
import pandas as pd
df = pd.read_csv(cfg['paths']['demographics_csv'])
print(df.groupby(['dataset','label']).size())
print(df['sex'].value_counts())
print(df['age_band'].value_counts())

## 6. Train + evaluate

In [ ]:
!python train.py --config config.yaml

## 7. Grad-CAM sanity check

In [ ]:
out_dir = cfg['paths']['out_dir']
!python gradcam.py --config config.yaml --checkpoint "{out_dir}/best.pt" --n 8

In [ ]:
# Display a couple of overlays.
import glob
from IPython.display import Image as IPImage, display
for p in sorted(glob.glob(f"{out_dir}/gradcam/*.png"))[:4]:
    display(IPImage(p))

## 8. Multi-seed run (mean ± 95% CI) — paper-ready

Runs the baseline over the seeds in `config.yaml` (`multiseed.seeds`) and aggregates each
metric as mean ± 95% CI. Writes `summary.csv`, `summary.md`, and `aggregate.json` into the
run folder — download those (plus `seed*/test_metrics.json`) into `experiments/results/`
and commit. This is what turns the single-seed fairness gaps into defensible claims.

In [ ]:
!python run_multiseed.py --config config.yaml

In [ ]:
# Show the aggregated paper-ready summary.
ms = cfg.get('multiseed', {})
exp = ms.get('exp_name', 'ex001_centralized_3seed')
runs_root = '/'.join(cfg['paths']['out_dir'].split('/')[:-1])
print(open(f"{runs_root}/{exp}/summary.md").read())